In [ ]:
import numpy as np
import pandas as pd
import hla_genes

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier


# Compute HLA allele counts per ethnicity

In [ ]:
hla_dosages_df = hla_genes.get_dosages_per_gene(create_hierarchical_index=True, debug_mode=False)
hla_dosages_df.index = hla_dosages_df.index.astype(int)

ETHNICITY_FILE = "/nfs/research/birney/controlled_access/ukb-cnv/data_fetch/baskets/2017133/self_reported_ethnicity_21000.txt"
ethnicity_df = pd.read_csv(ETHNICITY_FILE, sep='\t').set_index("f.eid").astype("Int64")

hla_dosages_df_long = hla_dosages_df.melt(ignore_index=False)
merged_df = ethnicity_df.merge(hla_dosages_df_long, left_index=True, right_index=True)
merged_df = merged_df.drop(["f.21000.1.0", "f.21000.2.0", "f.21000.3.0"], axis=1)
merged_df.columns = ["ethnicity", 'gene', 'serotype', 'subtype', 'value']

ethnicity_to_broad_mapping = { x: str(x)[0] for x in merged_df.ethnicity.unique() }
# broad_ethnicity = merged_df.ethnicity.apply(lambda x: ethnicity_to_broad_mapping.get(x, None))
# merged_df['broad_ethnicity'] = broad_ethnicity

# merged_df = merged_df[merged_df.broad_ethnicity.isin({str(i) for i in range(1, 7)})]
# merged_df.to_csv("hla2fields_long_with_ethnicity.csv")

In [ ]:
merged_df

In [ ]:
# counts = merged_df.drop("ethnicity",axis=1).groupby(["broad_ethnicity", "gene", "serotype", "subtype"]).value.sum()
counts = merged_df.groupby(["ethnicity", "gene", "serotype", "subtype"]).value.sum()
# counts.reset_index().pivot(index=["gene", "serotype", "subtype"], columns="broad_ethnicity", values="value").reset_index().to_csv("hla_4_digits_counts_per_ethnicity.csv")
hla_vs_ethinicity = counts.reset_index().pivot(index=["gene", "serotype", "subtype"], columns="ethnicity", values="value").reset_index()# .to_csv("hla_2_fields_counts_per_detailed_ethnicity.csv")

In [ ]:
hla_vs_ethinicity.columns = ["locus", "allele_groups", "subtype"] + hla_vs_ethinicity.columns[3:].to_list()
hla_vs_ethinicity.to_csv("hla_2_fields_counts_per_detailed_ethnicity.csv", index=False)

# Infer ethnicity from HLA alleles

In [ ]:
hla_dosages_df = hla_genes.get_dosages_per_gene(create_hierarchical_index=False, debug_mode=False)
hla_dosages_df.index = hla_dosages_df.index.astype(int)

In [ ]:
ETHNICITY_FILE = "/nfs/research/birney/controlled_access/ukb-cnv/data_fetch/baskets/2017133/self_reported_ethnicity_21000.txt"
ethnicity_df = pd.read_csv(ETHNICITY_FILE, sep='\t').set_index("f.eid").astype("Int64")
merged_df = ethnicity_df.merge(hla_dosages_df, left_index=True, right_index=True)

X = merged_df[merged_df.columns[merged_df.columns.str.contains("HLA")]]
y = merged_df['f.21000.0.0'].apply(lambda x: x if np.isnan(x) else str(x)[0])
y = y[y.apply(lambda x: x in {'1','3','4','5'})]
y.name = "ethnicity"

merged_df = pd.concat([y.to_frame(), X], axis=1).query('ethnicity != "-"')
min_count = merged_df["ethnicity"].value_counts().min()
merged_df = merged_df.groupby('ethnicity', group_keys=False).apply(lambda x: x.sample(min_count, random_state=42), include_groups=False)
merged_df = y.to_frame().merge(merged_df, left_index=True, right_index=True)

In [ ]:
ethnicity_codes = pd.read_csv("/homes/bonazzola/ukbb_helpers/codings/coding1001.tsv", sep='\t')[['coding', 'meaning']]
ethnicity_codes = ethnicity_codes[ethnicity_codes.coding.apply(lambda x: len(str(x)) == 1)]
ethnicity_codes.meaning = ethnicity_codes.meaning.apply(lambda x: x.split(" ")[0])

In [ ]:
X = merged_df.drop("ethnicity", axis=1)

In [ ]:
y = merged_df.ethnicity
y = (le := LabelEncoder()).fit_transform(y)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# clf = LogisticRegression(solver='lbfgs', max_iter=10000)
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

In [ ]:
y_pred = clf.predict(X_test)

In [ ]:
label_names = ethnicity_codes.meaning[[2,4,5,6]]

In [ ]:
print(classification_report(y_test, y_pred, target_names=label_names))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# y_true: etiquetas verdaderas
# y_pred: etiquetas predichas

# Generar matriz de confusión
cm = confusion_matrix(y_test, y_pred, labels=[0,1,2,3])

# Visualizar
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names,
            yticklabels=label_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
pca_df = pd.read_csv("/homes/bonazzola/ukbb_data/2017133/genetic_pcs_22009.txt", sep='\t').set_index("f.eid")
pca_df.columns = [f"PC{str(i+1).zfill(2)}" for i in range(40)]